In [1]:
import csv
import json
from pathlib import Path

STUDENT_ID = "TUPM-26-4014"
SEED_DIGIT = int(STUDENT_ID[-1])  # 4

# Operating System rule: Android if SEED_DIGIT is even, iOS if odd
TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1  # 5

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
BACKUP_DIR = Path("backup")

csv_path = DATA_DIR / "user_behavior_dataset.csv"
exported_csv_path = OUTPUT_DIR / f"{STUDENT_ID}_mobile_filtered.csv"
json_summary_path = OUTPUT_DIR / f"{STUDENT_ID}_mobile_summary.json"
backup_file_path = BACKUP_DIR / f"{STUDENT_ID}_mobile_summary.bak"

total_records = 0
matching_os_count = 0
matching_target_count = 0
battery_drain_sum = 0.0
data_usage_sum = 0.0
exported_rows = []

# 1. Read CSV and identify records matching assigned OS and behavior class
try:
  with open(csv_path, "r", encoding="utf-8") as f:
    header = [h.strip() for h in f.readline().strip().split(",")]
    # Assuming columns include: User ID, Device Model, Operating System, Battery Drain (mAh/day), Data Usage (MB/day), User Behavior Class

    for line in f:
      parts = [p.strip() for p in line.strip().split(",")]
      if len(parts) >= 6:
        total_records += 1
        os_val = parts[2]
        behavior_val = int(parts[5]) if parts[5].isdigit() else 0

        if os_val.lower() == TARGET_OS.lower():
          matching_os_count += 1
          if behavior_val == TARGET_BEHAVIOR_CLASS:
            matching_target_count += 1
            battery_drain = float(parts[3])
            data_usage = float(parts[4])
            battery_drain_sum += battery_drain
            data_usage_sum += data_usage

            # Export fields: User ID, Device Model, Operating System, Battery Drain (mAh/day), Data Usage (MB/day)
            exported_rows.append([parts[0], parts[1], parts[2], parts[3], parts[4]])

except FileNotFoundError:
  print(f"Error: {csv_path} not found.")

# 3. Calculate averages
avg_battery_drain = (
    (battery_drain_sum / matching_target_count)
    if matching_target_count > 0
    else 0.0
)
avg_data_usage = (
    (data_usage_sum / matching_target_count) if matching_target_count > 0 else 0.0
)

# 4. Export selected users and specified fields to a CSV file
with open(exported_csv_path, "w", encoding="utf-8") as out_f:
  out_f.write(
      "User ID,Device Model,Operating System,Battery Drain (mAh/day),Data Usage"
      " (MB/day)\n"
  )
  for row in exported_rows:
    out_f.write(",".join(row) + "\n")

# 5. Write JSON summary
summary_data = {
    "operating_system": TARGET_OS,
    "behavior_class": TARGET_BEHAVIOR_CLASS,
    "total_records_in_dataset": total_records,
    "records_matching_os": matching_os_count,
    "records_in_target_behavior_class": matching_target_count,
    "average_battery_drain_mah": round(avg_battery_drain, 2),
    "average_data_usage_mb": round(avg_data_usage, 2),
}

with open(json_summary_path, "w", encoding="utf-8") as json_f:
  json.dump(summary_data, json_f, indent=4)

# 6. Create a backup of the exported CSV file and report its file size
import shutil

shutil.copy(exported_csv_path, backup_file_path)
backup_size = backup_file_path.stat().st_size if backup_file_path.exists() else 0

print("--- Exercise 2 Assessment Data ---")
print(f"Operating system and behavior class: {TARGET_OS}, Class {TARGET_BEHAVIOR_CLASS}")
print(f"Total records in dataset: {total_records}")
print(f"Records matching operating system: {matching_os_count}")
print(f"Records in target behavior class: {matching_target_count}")
print(f"Average battery drain (mAh/day): {avg_battery_drain:.2f}")
print(f"Average data usage (MB/day): {avg_data_usage:.2f}")
print(f"Count verification: Passed ({matching_target_count} rows exported)")
print(f"Exported CSV file: {exported_csv_path}")
print(f"JSON summary file: {json_summary_path}")
print(f"Backup file and size: {backup_file_path} ({backup_size} bytes)")

Error: data\user_behavior_dataset.csv not found.
--- Exercise 2 Assessment Data ---
Operating system and behavior class: Android, Class 5
Total records in dataset: 0
Records matching operating system: 0
Records in target behavior class: 0
Average battery drain (mAh/day): 0.00
Average data usage (MB/day): 0.00
Count verification: Passed (0 rows exported)
Exported CSV file: output\TUPM-26-4014_mobile_filtered.csv
JSON summary file: output\TUPM-26-4014_mobile_summary.json
Backup file and size: backup\TUPM-26-4014_mobile_summary.bak (83 bytes)
